# Task 2 — Evaluasi Baseline pada Warm Validation

Membandingkan **Popularity**, **Item-CF**, dan **TF-IDF CBF** menggunakan protokol `Task_1_Evaluation_Setup.ipynb`. Implementasi model mengikuti `Task_1_Experimental_Protocol(1).ipynb` dan modul proyek yang sama.

**Cara menjalankan:** simpan notebook ini di folder repository yang berisi `src/`, `MAIN DATA-...`, dan `SUPPORTING_DATA-...`. Pilih kernel Python dengan dependensi proyek, lalu **Restart Kernel and Run All**. Notebook tidak memerlukan Task 1 dijalankan pada kernel yang sama. Artefak hasil ditulis ke `outputs/task2/`.

Tidak ada angka hasil eksperimen yang diisi sebelum notebook dijalankan. Data dan `src/` harus tersedia di proyek lokal. Jangan mengganti tiga baseline atau mengubah parameter di tengah perbandingan.


## 1. Setup dan protokol Task 1

In [ ]:
from pathlib import Path
import json
import math
import sys

import numpy as np
import pandas as pd

# Cari root repository yang memiliki src/data.py.
PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "src" / "data.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Root proyek dengan src/data.py tidak ditemukan.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import load_project_data

K = 10
POSITIVE_THRESHOLD = 4.0
print("Project root:", PROJECT_ROOT)

In [ ]:
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from IPython.display import display
from src.recommenders import (
    fit_popularity, fit_item_cf, recommend_popularity,
    recommend_from_model, _prepare_matrices,
)
TOP_N = K
TOP_K_SIMILARITY = 100
CHUNK_SIZE = 32


c:\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


NameError: name 'K' is not defined

In [ ]:
data = load_project_data(PROJECT_ROOT)
train = data.train.copy()
items = data.items.copy()
validation_warm = data.validation_warm.copy()
candidates = list(dict.fromkeys(str(item_id) for item_id in data.candidates))
eval_users = list(dict.fromkeys(str(user_id) for user_id in data.evaluable_users))

for frame in (train, validation_warm):
    frame["user_id"] = frame["user_id"].astype(str)
    frame["item_id"] = frame["item_id"].astype(str)
items["item_id"] = items["item_id"].astype(str)

candidate_set = set(candidates)
eval_user_set = set(eval_users)
train_user_set = set(train["user_id"])
train_item_set = set(train["item_id"])
metadata_item_set = set(items["item_id"])

training_user_count = train["user_id"].nunique()
training_item_count = train["item_id"].nunique()
print(f"Training interactions : {len(train):,}")
print(f"Training users        : {training_user_count:,}")
print(f"Training items        : {training_item_count:,}")
print(f"Warm validation rows  : {len(validation_warm):,}")
print(f"Evaluable users       : {len(eval_users):,}")
print(f"Candidate universe    : {len(candidates):,}")

In [ ]:
assert eval_users and candidates, "Daftar user atau kandidat kosong."
assert not train[["user_id", "item_id", "rating"]].isna().any().any()
assert not validation_warm[["user_id", "item_id", "rating"]].isna().any().any()
assert not items["item_id"].duplicated().any(), "Metadata item_id harus unik."


In [ ]:
assert eval_user_set <= train_user_set, (
    f"Ada {len(eval_user_set - train_user_set):,} evaluable users tanpa histori training."
)
assert len(candidates) == len(candidate_set), "Candidate universe mengandung duplikat."
assert candidate_set <= train_item_set, "Ada kandidat yang tidak muncul di training."
assert candidate_set <= metadata_item_set, "Ada kandidat tanpa metadata item."

seen_by_user = {
    user_id: set(group["item_id"])
    for user_id, group in train.groupby("user_id", sort=False)
}

print("Audit evaluable users       : PASS")
print("Audit candidate universe    : PASS")
print("Candidate items in metadata: PASS")
print("Candidate items in training : PASS")

## 2. Ground truth yang sama untuk semua baseline

Gunakan validation warm resmi, user resmi, rating ≥ 4, dan item dalam candidate universe. Hapus target yang sudah dilihat pada training. Semua user resmi tetap menjadi denominator macro-average; user tanpa target diberi nilai nol, sesuai Task 1. Tidak membaca Final Test dan tidak membuat split baru.


In [ ]:
validation_eval = validation_warm[
    validation_warm["user_id"].isin(eval_user_set)
].copy()

positive_validation = validation_eval.loc[
    (validation_eval["rating"] >= POSITIVE_THRESHOLD)
    & validation_eval["item_id"].isin(candidate_set)
].copy()

positive_validation["training_seen"] = [
    item_id in seen_by_user.get(user_id, set())
    for user_id, item_id in zip(
        positive_validation["user_id"],
        positive_validation["item_id"],
    )
]

relevant_rows = positive_validation.loc[
    ~positive_validation["training_seen"]
].copy()
relevant_by_user = {
    user_id: set(group["item_id"])
    for user_id, group in relevant_rows.groupby("user_id", sort=False)
}
for user_id in eval_users:
    relevant_by_user.setdefault(user_id, set())

users_without_target = [
    user_id for user_id in eval_users
    if not relevant_by_user[user_id]
]

print(f"Validation rows after user filter       : {len(validation_eval):,}")
print(f"Positive candidate validation rows      : {len(positive_validation):,}")
seen_target_count = int(positive_validation["training_seen"].sum())
print(f"Training-seen positive targets excluded : {seen_target_count:,}")
print(f"Relevant target rows                    : {len(relevant_rows):,}")
print(f"Users without relevant target           : {len(users_without_target):,}")

assert set(relevant_by_user) == eval_user_set
assert all(targets <= candidate_set for targets in relevant_by_user.values())
assert all(not (targets & seen_by_user.get(user_id, set()))
           for user_id, targets in relevant_by_user.items())

## 3. Metrik dan audit dari Task 1

In [ ]:
def recall_at_k(ranked_items, relevant_items, k=K):
    ranked = [str(item_id) for item_id in ranked_items[:k]]
    targets = set(map(str, relevant_items))
    if not targets:
        return 0.0
    if len(ranked) != len(set(ranked)):
        raise ValueError("Ranking mengandung item duplikat.")
    return len(set(ranked) & targets) / len(targets)

def ndcg_at_k(ranked_items, relevant_items, k=K):
    ranked = [str(item_id) for item_id in ranked_items[:k]]
    targets = set(map(str, relevant_items))
    if not targets:
        return 0.0
    if len(ranked) != len(set(ranked)):
        raise ValueError("Ranking mengandung item duplikat.")
    dcg = sum(
        1.0 / math.log2(position + 2)
        for position, item_id in enumerate(ranked)
        if item_id in targets
    )
    ideal_count = min(k, len(targets))
    idcg = sum(
        1.0 / math.log2(position + 2)
        for position in range(ideal_count)
    )
    return dcg / idcg if idcg else 0.0

# Sanity checks matematis.
truth = {"A", "B"}
assert recall_at_k(["A", "B", "X"], truth) == 1.0
assert math.isclose(ndcg_at_k(["A", "B", "X"], truth), 1.0)
assert recall_at_k(["X", "A", "B"], truth) == 1.0
assert ndcg_at_k(["X", "A", "B"], truth) < 1.0
assert recall_at_k(["X", "Y"], truth) == 0.0
assert ndcg_at_k(["X", "Y"], truth) == 0.0
print("Sanity checks Recall@10/NDCG@10: PASS")

In [ ]:
def extract_item_ids(ranking):
    result = []
    for entry in ranking:
        result.append(str(entry[0]) if isinstance(entry, (tuple, list)) else str(entry))
    return result

def evaluate_recommendations(recommendations, model_name="model", k=K):
    recommendation_users = set(map(str, recommendations))
    if recommendation_users != eval_user_set:
        raise AssertionError(
            f"{model_name}: user universe berbeda dari evaluable users."
        )

    rows = []
    for user_id in eval_users:
        ranked = extract_item_ids(recommendations[user_id])
        if len(ranked) > k:
            raise AssertionError(f"{model_name}: lebih dari Top-{k} item.")
        if len(ranked) != len(set(ranked)):
            raise AssertionError(f"{model_name}: ranking memiliki duplikasi.")
        if not set(ranked) <= candidate_set:
            raise AssertionError(f"{model_name}: ada item di luar candidate universe.")
        if any(item_id in seen_by_user.get(user_id, set()) for item_id in ranked):
            raise AssertionError(f"{model_name}: ada item yang sudah dilihat user.")

        targets = relevant_by_user[user_id]
        rows.append({
            "model": model_name,
            "user_id": user_id,
            "n_relevant": len(targets),
            "n_recommended": len(ranked),
            f"recall@{k}": recall_at_k(ranked, targets, k),
            f"ndcg@{k}": ndcg_at_k(ranked, targets, k),
        })

    per_user = pd.DataFrame(rows)
    summary = {
        "model": model_name,
        "users_evaluated": len(per_user),
        f"recall@{k}": float(per_user[f"recall@{k}"].mean()),
        f"ndcg@{k}": float(per_user[f"ndcg@{k}"].mean()),
    }
    return summary, per_user

In [ ]:
def audit_complete_top10(recommendations, model_name):
    # Task 1 mengizinkan ranking pendek; Task 2 juga mencatat dan memeriksa
    # kelengkapan Top-10 bila kandidat unseen masih mencukupi.
    short_users = 0
    for user_id in eval_users:
        expected = min(K, len(candidate_set - seen_by_user.get(user_id, set())))
        actual = len(recommendations[user_id])
        if actual != expected:
            raise AssertionError(
                f"{model_name}: user {user_id} menerima {actual} item; "
                f"seharusnya {expected}. Periksa fallback di src.recommenders."
            )
        short_users += actual < K
    print(f"{model_name}: audit jumlah rekomendasi PASS; user <{K}: {short_users}")

summaries = []
per_user_results = []
all_recommendations = {}

def record_result(name, recs):
    # Audit kandidat, seen item, duplikasi, dan user dilakukan sebelum metrik.
    summary, per_user = evaluate_recommendations(recs, model_name=name, k=K)
    audit_complete_top10(recs, name)
    all_recommendations[name] = recs
    summaries.append(summary)
    per_user_results.append(per_user)
    display(pd.DataFrame([summary]))


## 4. Popularity

Menghitung interaksi positif pada training (rating ≥ 4). Kandidat sama untuk seluruh user, dengan item yang sudah dilihat dikeluarkan per user. Menggunakan fungsi baseline proyek, seperti notebook Experimental Protocol.


In [ ]:
popularity = fit_popularity(train, positive_threshold=POSITIVE_THRESHOLD)
pop_recs = recommend_popularity(
    eval_users, candidates, popularity, seen_by_user, top_n=TOP_N
)
record_result("Popularity", pop_recs)


## 5. Item-CF

Fit hanya dari training dengan `top_k=100` dan rating positif ≥ 4. Fungsi rekomendasi proyek menerima popularity sebagai fallback. Semua hasil diperiksa dengan evaluator Task 1 yang sama.


In [ ]:
item_cf_model = fit_item_cf(
    train, candidates, top_k=TOP_K_SIMILARITY,
    positive_threshold=POSITIVE_THRESHOLD,
)
item_cf_recs = recommend_from_model(
    item_cf_model, eval_users, candidates, popularity, top_n=TOP_N
)
record_result("Item-CF", item_cf_recs)


## 6. TF-IDF Content-Based Filtering

Mengikuti implementasi berbasis `content_text_basic` pada notebook Experimental Protocol: TF-IDF unigram/bigram, maksimal 100.000 fitur, dan 100 tetangga per item. Kemiripan dihitung per blok agar penggunaan RAM lebih kecil. Validation ground truth tidak digunakan untuk membangun model. Metadata dibatasi ke kandidat resmi yang muncul pada training.

Jika RAM tidak cukup, turunkan `CHUNK_SIZE` menjadi 16 atau 8 lalu jalankan ulang. Ukuran blok hanya mengatur komputasi, bukan parameter model.


In [ ]:
assert "content_text_basic" in items.columns, (
    "Kolom content_text_basic tidak ditemukan. Periksa nama kolom pada dataset/modul."
)

# Bangun representasi TF-IDF seperti implementasi src, tetapi hitung kemiripan
# per blok agar tidak membuat matriks item-item besar sekaligus di RAM.
user_ids, item_ids, user_index, item_index, interactions, seen = _prepare_matrices(
    train, candidates, POSITIVE_THRESHOLD
)
metadata = items.loc[items["item_id"].isin(candidate_set)].copy()
metadata["item_id"] = metadata["item_id"].astype(str)
text_by_id = metadata.set_index("item_id")["content_text_basic"].fillna("").astype(str)
documents = [text_by_id.get(item_id, "") for item_id in item_ids]

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.8,
    max_features=100_000,
    sublinear_tf=True,
    dtype=np.float32,
)
item_vectors = vectorizer.fit_transform(documents).tocsr()

# Top-100 tetangga per item. Ubah CHUNK_SIZE menjadi 16 jika RAM masih terbatas.
TOP_K_SIMILARITY = 100
CHUNK_SIZE = 32
n_items = item_vectors.shape[0]
rows, cols, values = [], [], []

for start in range(0, n_items, CHUNK_SIZE):
    stop = min(start + CHUNK_SIZE, n_items)
    block_similarity = (item_vectors[start:stop] @ item_vectors.T).tocsr()
    block_similarity.eliminate_zeros()

    for local_row in range(stop - start):
        global_row = start + local_row
        row_start = block_similarity.indptr[local_row]
        row_end = block_similarity.indptr[local_row + 1]
        row_cols = block_similarity.indices[row_start:row_end]
        row_values = block_similarity.data[row_start:row_end]

        # Jangan masukkan item itu sendiri sebagai tetangga.
        valid = row_cols != global_row
        row_cols, row_values = row_cols[valid], row_values[valid]
        if len(row_values) > TOP_K_SIMILARITY:
            keep = np.argpartition(row_values, -TOP_K_SIMILARITY)[-TOP_K_SIMILARITY:]
            row_cols, row_values = row_cols[keep], row_values[keep]

        rows.extend([global_row] * len(row_cols))
        cols.extend(row_cols.tolist())
        values.extend(row_values.astype(np.float32).tolist())

    del block_similarity

similarity_graph = sparse.csr_matrix(
    (np.asarray(values, dtype=np.float32),
     (np.asarray(rows, dtype=np.int32), np.asarray(cols, dtype=np.int32))),
    shape=(n_items, n_items),
    dtype=np.float32,
)

tfidf_cbf_model = {
    "user_ids": user_ids,
    "item_ids": item_ids,
    "user_index": user_index,
    "item_index": item_index,
    "interactions": interactions,
    "seen": seen,
    "similarity": similarity_graph,
    "vectorizer": vectorizer,
}

tfidf_cbf_recs = recommend_from_model(
    tfidf_cbf_model, eval_users, candidates, popularity, top_n=TOP_N
)

record_result("TF-IDF CBF", tfidf_cbf_recs)


## 7. Perbandingan hasil validation

In [ ]:
results = pd.DataFrame(summaries)
assert len(results) == 3
assert results["users_evaluated"].eq(len(eval_users)).all()
assert results[["recall@10", "ndcg@10"]].ge(0).all().all()
assert results[["recall@10", "ndcg@10"]].le(1 + 1e-12).all().all()
display(results.set_index("model").style.format({"recall@10": "{:.6f}", "ndcg@10": "{:.6f}"}))

for metric in ("recall@10", "ndcg@10"):
    best = results[metric].max()
    winners = results.loc[np.isclose(results[metric], best, rtol=0, atol=1e-12), "model"].tolist()
    print(f"Tertinggi pada {metric}: {', '.join(winners)} ({best:.6f})")
print("Hasil ini berlaku untuk warm validation dengan protokol ini; bukan hasil Final Test.")


### Cara menulis analisis

- Recall@10 lebih tinggi berarti proporsi target relevan yang ditemukan lebih besar.
- NDCG@10 lebih tinggi berarti target relevan mendapat posisi lebih baik, dengan normalisasi terhadap ranking ideal.
- Jika pemenangnya berbeda, jelaskan perbedaan kedua metrik; jangan memaksakan satu model unggul pada semua aspek.
- Tidak ada batas minimum keberhasilan yang diberikan. Selisih skor bukan bukti signifikansi statistik.
- Nilai Popularity pada keluaran Task 1 yang dilampirkan adalah Recall@10 sekitar 0,013499 dan NDCG@10 sekitar 0,009222. Itu acuan pengecekan, bukan hasil Task 2 yang dijamin sama: periksa implementasi dan tie-breaking jika berbeda.


## 8. Inspeksi Top-10 satu user

In [ ]:
sample_user = eval_users[0]
print("User:", sample_user)
print("Target relevan:", sorted(relevant_by_user[sample_user]))
for name, recs in all_recommendations.items():
    ids = extract_item_ids(recs[sample_user])
    sample = pd.DataFrame({"rank": range(1, len(ids) + 1), "item_id": ids})
    sample["relevant"] = sample["item_id"].isin(relevant_by_user[sample_user])
    if "title" in items.columns:
        sample = sample.merge(items[["item_id", "title"]], on="item_id", how="left", sort=False)
    print(name)
    display(sample)


## 9. Simpan hasil yang dapat diperiksa

In [ ]:
output_dir = PROJECT_ROOT / "outputs" / "task2"
output_dir.mkdir(parents=True, exist_ok=True)
results.to_csv(output_dir / "baseline_comparison.csv", index=False)
pd.concat(per_user_results, ignore_index=True).to_csv(
    output_dir / "baseline_metrics_per_user.csv", index=False
)
recommendation_rows = []
for name, recs in all_recommendations.items():
    for user_id in eval_users:
        for rank, item_id in enumerate(extract_item_ids(recs[user_id]), start=1):
            recommendation_rows.append({"model": name, "user_id": user_id,
                                        "rank": rank, "item_id": item_id})
pd.DataFrame(recommendation_rows).to_csv(output_dir / "baseline_top10.csv", index=False)

import hashlib
import inspect

def id_digest(values):
    return hashlib.sha256(json.dumps(sorted(values)).encode("utf-8")).hexdigest()

protocol = {
    "task": "Task 2 - Baseline Evaluation",
    "protocol_source": "Task_1_Evaluation_Setup.ipynb",
    "model_source": "Task_1_Experimental_Protocol(1).ipynb + src.recommenders",
    "split": "official train + warm validation", "uses_final_test": False,
    "train_interactions": len(train), "warm_validation_rows": len(validation_warm),
    "evaluable_users": len(eval_users), "candidate_items": len(candidates),
    "users_without_target": len(users_without_target),
    "seen_positive_targets_excluded": int(positive_validation["training_seen"].sum()),
    "relevant_rows": len(relevant_rows), "positive_threshold": POSITIVE_THRESHOLD,
    "k": K, "aggregation": "macro average over all official evaluable users",
    "candidate_ids_sha256": id_digest(candidates),
    "eval_user_ids_sha256": id_digest(eval_users),
    "item_cf_top_k": TOP_K_SIMILARITY, "tfidf_top_k": TOP_K_SIMILARITY,
    "tfidf_parameters": vectorizer.get_params(), "chunk_size": CHUNK_SIZE,
    "recommenders_source_sha256": hashlib.sha256(
        Path(inspect.getfile(fit_item_cf)).read_bytes()).hexdigest(),
    "audit": "PASS", "results": results.to_dict(orient="records"),
}
# Parameter dtype ditulis sebagai string agar dapat disimpan dalam JSON.
protocol["tfidf_parameters"]["dtype"] = str(protocol["tfidf_parameters"]["dtype"])
(output_dir / "task2_protocol_and_results.json").write_text(
    json.dumps(protocol, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Selesai. Hasil tersimpan di:", output_dir)
